<a href="https://colab.research.google.com/github/huyd073003/AAI2026/blob/inventory_replenishment_agent/inventory_replenishment_agent_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import files
uploaded = files.upload()

Saving params.csv to params.csv


In [5]:
import pandas as pd
import numpy as np
from math import sqrt
from statistics import NormalDist

sales = pd.read_csv('sales.csv')
inventory = pd.read_csv('inventory.csv')
params = pd.read_csv('params.csv')

sales['date'] = pd.to_datetime(sales['date'])

print('sales')
display(sales.head())

print('inventory')
display(inventory.head())

print('params')
display(params.head())

sales


,date,sku,qty_sold
0,2025-01-01,SKU_A,30
1,2025-01-01,SKU_B,24
2,2025-01-01,SKU_C,18
3,2025-01-02,SKU_A,35
4,2025-01-02,SKU_B,21


inventory


,sku,opening_stock
0,SKU_A,180
1,SKU_B,130
2,SKU_C,100


params


,sku,unit_cost,holding_cost_per_day,stockout_cost,lead_time_days,min_order_qty,service_level
0,SKU_A,14.0,0.040,11.0,4,35,0.95
1,SKU_B,9.0,0.030,8.0,3,30,0.92
2,SKU_C,6.5,0.025,6.0,2,25,0.90


In [6]:
def run_policy(sales, inventory, params, use_ewma=True, alpha=0.35):
    dates = sorted(sales['date'].unique())
    skus = sorted(params['sku'].unique())

    opening_stock = dict(zip(inventory['sku'], inventory['opening_stock']))
    param_map = params.set_index('sku').to_dict('index')

    state = {}
    for sku in skus:
        sku_sales = sales[sales['sku'] == sku].sort_values('date')
        initial_forecast = max(1, sku_sales['qty_sold'].iloc[:7].mean())
        state[sku] = {
            'on_hand': float(opening_stock[sku]),
            'forecast': float(initial_forecast),
            'errors': [],
            'pipeline': []
        }

    logs = []

    for current_date in dates:
        day_sales = sales[sales['date'] == current_date]

        for sku in skus:
            p = param_map[sku]
            st = state[sku]

            arrivals = sum(qty for arrival_date, qty in st['pipeline'] if arrival_date == current_date)
            st['pipeline'] = [(d, q) for d, q in st['pipeline'] if d != current_date]
            st['on_hand'] += arrivals

            actual_row = day_sales[day_sales['sku'] == sku]
            actual_demand = float(actual_row['qty_sold'].sum()) if not actual_row.empty else 0.0

            demand_over_lead = st['forecast'] * p['lead_time_days']

            if len(st['errors']) >= 2:
                sigma = np.std(st['errors'], ddof=1)
            elif len(st['errors']) == 1:
                sigma = abs(st['errors'][0])
            else:
                sigma = max(1.0, st['forecast'] * 0.2)

            z = NormalDist().inv_cdf(p['service_level'])
            safety_stock = z * sigma * sqrt(p['lead_time_days'])
            reorder_point = demand_over_lead + safety_stock

            inventory_position = st['on_hand'] + sum(q for _, q in st['pipeline'])

            target_level = demand_over_lead + safety_stock + st['forecast'] * 2
            raw_order_qty = max(0, target_level - inventory_position)

            if inventory_position <= reorder_point:
                order_qty = max(p['min_order_qty'], int(np.ceil(raw_order_qty)))
                order_qty = min(order_qty, int(np.ceil(st['forecast'] * (p['lead_time_days'] + 10))))
                arrival_date = current_date + pd.Timedelta(days=int(p['lead_time_days']))
                st['pipeline'].append((arrival_date, order_qty))
                action = 'ORDER'
                rationale = f'Projected inventory below reorder point; ordered {order_qty}'
            else:
                order_qty = 0
                action = 'WAIT'
                rationale = 'Inventory position above reorder point'

            fulfilled = min(st['on_hand'], actual_demand)
            stockout = max(0, actual_demand - fulfilled)
            ending_on_hand = st['on_hand'] - fulfilled

            holding_cost = ending_on_hand * p['holding_cost_per_day']
            stockout_cost = stockout * p['stockout_cost']
            total_cost = holding_cost + stockout_cost

            logs.append({
                'date': current_date,
                'sku': sku,
                'begin_on_hand': round(st['on_hand'], 2),
                'arrivals': round(arrivals, 2),
                'forecast': round(st['forecast'], 2),
                'actual_demand': round(actual_demand, 2),
                'safety_stock': round(safety_stock, 2),
                'reorder_point': round(reorder_point, 2),
                'inventory_position': round(inventory_position, 2),
                'action': action,
                'order_qty': round(order_qty, 2),
                'fulfilled': round(fulfilled, 2),
                'stockout_units': round(stockout, 2),
                'ending_on_hand': round(ending_on_hand, 2),
                'holding_cost': round(holding_cost, 2),
                'stockout_cost': round(stockout_cost, 2),
                'total_cost': round(total_cost, 2),
                'rationale': rationale
            })

            error = actual_demand - st['forecast']
            st['errors'].append(error)
            st['on_hand'] = ending_on_hand

            if use_ewma:
                st['forecast'] = alpha * actual_demand + (1 - alpha) * st['forecast']
            else:
                st['forecast'] = max(1.0, actual_demand)

    log_df = pd.DataFrame(logs)

    summary = (
        log_df.groupby('sku')
        .agg(
            total_demand=('actual_demand', 'sum'),
            total_fulfilled=('fulfilled', 'sum'),
            total_stockouts=('stockout_units', 'sum'),
            total_holding_cost=('holding_cost', 'sum'),
            total_stockout_cost=('stockout_cost', 'sum'),
            total_cost=('total_cost', 'sum')
        )
        .reset_index()
    )

    summary['fill_rate'] = np.where(
        summary['total_demand'] > 0,
        summary['total_fulfilled'] / summary['total_demand'],
        1.0
    )

    overall = pd.DataFrame([{
        'sku': 'ALL',
        'total_demand': summary['total_demand'].sum(),
        'total_fulfilled': summary['total_fulfilled'].sum(),
        'total_stockouts': summary['total_stockouts'].sum(),
        'total_holding_cost': summary['total_holding_cost'].sum(),
        'total_stockout_cost': summary['total_stockout_cost'].sum(),
        'total_cost': summary['total_cost'].sum(),
        'fill_rate': summary['total_fulfilled'].sum() / summary['total_demand'].sum()
    }])

    summary = pd.concat([summary, overall], ignore_index=True)

    return log_df, summary

In [7]:
agent_log, agent_summary = run_policy(sales, inventory, params, use_ewma=True, alpha=0.35)
baseline_log, baseline_summary = run_policy(sales, inventory, params, use_ewma=False, alpha=0.35)

print('Agent summary')
display(agent_summary)

print('Baseline summary')
display(baseline_summary)

Agent summary


,sku,total_demand,total_fulfilled,total_stockouts,total_holding_cost,total_stockout_cost,total_cost,fill_rate
0,SKU_A,3163.0,2925.0,238.0,170.76,2618.0,2788.76,0.924755
1,SKU_B,2051.0,1897.0,154.0,74.43,1232.0,1306.43,0.924915
2,SKU_C,1403.0,1242.0,161.0,37.55,966.0,1003.55,0.885246
3,ALL,6617.0,6064.0,553.0,282.74,4816.0,5098.74,0.916427


Baseline summary


,sku,total_demand,total_fulfilled,total_stockouts,total_holding_cost,total_stockout_cost,total_cost,fill_rate
0,SKU_A,3163.0,2886.0,277.0,259.80,3047.0,3306.80,0.912425
1,SKU_B,2051.0,1919.0,132.0,92.04,1056.0,1148.04,0.935641
2,SKU_C,1403.0,1297.0,106.0,43.22,636.0,679.22,0.924448
3,ALL,6617.0,6102.0,515.0,395.06,4739.0,5134.06,0.922170


In [8]:
agent_all = agent_summary[agent_summary['sku'] == 'ALL'].iloc[0]
baseline_all = baseline_summary[baseline_summary['sku'] == 'ALL'].iloc[0]

comparison = pd.DataFrame([
    {
        'strategy': 'Agent (EWMA)',
        'stockouts': agent_all['total_stockouts'],
        'fill_rate': round(agent_all['fill_rate'], 4),
        'total_cost': round(agent_all['total_cost'], 2)
    },
    {
        'strategy': 'Baseline (naive)',
        'stockouts': baseline_all['total_stockouts'],
        'fill_rate': round(baseline_all['fill_rate'], 4),
        'total_cost': round(baseline_all['total_cost'], 2)
    }
])

display(comparison)

,strategy,stockouts,fill_rate,total_cost
0,Agent (EWMA),553.0,0.9164,5098.74
1,Baseline (naive),515.0,0.9222,5134.06


In [9]:
agent_log.to_csv('daily_agent_log.csv', index=False)
baseline_log.to_csv('daily_baseline_log.csv', index=False)
agent_summary.to_csv('agent_summary.csv', index=False)
baseline_summary.to_csv('baseline_summary.csv', index=False)
comparison.to_csv('evaluation_comparison.csv', index=False)

print('Saved files:')
print('- daily_agent_log.csv')
print('- daily_baseline_log.csv')
print('- agent_summary.csv')
print('- baseline_summary.csv')
print('- evaluation_comparison.csv')

Saved files:
- daily_agent_log.csv
- daily_baseline_log.csv
- agent_summary.csv
- baseline_summary.csv
- evaluation_comparison.csv


In [10]:
files.download('daily_agent_log.csv')
files.download('evaluation_comparison.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Conclusion

The agent uses EWMA demand forecasting, safety stock, service level, and lead time to decide when to place replenishment orders. It balances the trade-off between avoiding stockouts and controlling holding cost. The daily log makes the decisions transparent, which supports trust and easier review of agent actions.